<a href="https://colab.research.google.com/github/haida-ishtiaq/FlyRankAI-ML-Internship/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [21]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [22]:
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import GroupShuffleSplit, ShuffleSplit
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
print(os.listdir("/content/drive/MyDrive/work/outputs"))

['baseline_action_score.csv', '_train_idx.npy', '_test_idx.npy', 'model_vs_baseline.csv', 'split_before_after.csv', 'content_action_queue.csv', 'playbook_metrics.json']


In [23]:

DRIVE_OUTPUTS = "/content/drive/MyDrive/work/outputs"
os.makedirs(DRIVE_OUTPUTS, exist_ok=True)

# writing:
pd.DataFrame({...}).to_csv(f"{DRIVE_OUTPUTS}/split_before_after.csv", index=False)

# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/haida-ishtiaq/FlyRankAI-ML-Internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

**Finding 1 -- "Random forest beats the hand-written baseline in 7 of 8 client-grouped splits (precision@50 0.805 vs 0.605)."** (from the published capstone paper)

**My methodology question:** with only 8 splits, is a 7-of-8 win count statistically distinguishable from a coin flip, given the paper's own overlapping standard deviations (forest sd 0.108, rule sd 0.114)? A binomial test on 7/8 against a 50/50 null gives p is approximately 0.07 -- suggestive, not conventionally significant. Separately, and specific to this repo: **my own re-run of the equivalent comparison on this project's actual starter-CSV data (`w05_model.ipynb`) shows the opposite ordering for Random Forest** -- it underperforms both the base rate and the rule baseline at K=20 and K=50 (Section 2 below). That doesn't mean the paper's number is wrong; the paper may be built on the larger warehouse release rather than the 30k-row starter CSV, and the two datasets are not interchangeable. But it does mean the specific claim "Random Forest is the stronger method" should not be assumed to transfer across datasets in this repo without being checked in each one, which is exactly what this notebook does in Section 2.

**Finding 2 -- "The model is weakest where it matters most: accuracy 0.57 on the largest-traffic tier, and the top-50 queue caught 0 of 259 declining pages in that tier."** (from the published capstone paper)

**My methodology question:** was the 0-of-259 figure evaluated against that tier's own top-K, or only within a global top-50 shared across all tiers? If the latter, "0 caught" could partly reflect other tiers' pages structurally outscoring high-traffic pages for shared top-50 slots, rather than the model having zero signal within that tier specifically. Section 2 below runs the equivalent subgroup check on this repo's own validated model (Logistic Regression, not Random Forest) so the answer is checked here, not assumed from the paper.

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

`w05_model.ipynb` already used a client-grouped split (`GroupShuffleSplit` on `client_id`, `test_size=0.2`, seed 42) and its own comparison table shows **Logistic Regression as the clear winner** -- so that is the model re-run here, at the same split ratio as w05, not a different one. Before = a naive random row split; after = the same client-grouped design w05 used. Per `hunting-leakage-and-validating/SKILL.md`: "Report the random-split number next to the honest-split number... the GAP between them is itself a finding about how much memorization was happening."

In [30]:


if not os.path.exists("data/raw/content_refresh_anonymized.csv"):
    os.chdir("/content")
    if not os.path.isdir("FlyRankAI-ML-Internship"):
        import subprocess
        subprocess.run([
            "git", "clone", "--depth", "1",
            "https://github.com/haida-ishtiaq/FlyRankAI-ML-Internship"
        ], check=True)
    os.chdir("FlyRankAI-ML-Internship")

RANDOM_SEED = 42
TEST_SIZE = 0.2  # matches w05_model.ipynb exactly, not a different ratio
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# --- Identical feature pipeline to w05_model.ipynb (same raw_numeric list, same has_* logic) ---
def build_features(df, train_idx_for_median):
    work = df.copy()
    work["has_position_data"] = (work["avg_position"] != 0).astype(int)
    work["avg_position"] = work["avg_position"].replace(0, np.nan)

    raw_numeric = ["impressions_90d", "ctr", "avg_position", "engagement_rate",
                   "scroll_rate", "days_since_last_update", "content_age_days",
                   "word_count", "days_with_impressions"]

    numeric_feats = ["has_position_data"]
    for col in raw_numeric:
        if work[col].isnull().any():
            work[f"has_{col}"] = work[col].notna().astype(int)
            train_median = work.iloc[train_idx_for_median][col].median()
            work[f"{col}_filled"] = work[col].fillna(train_median)
            numeric_feats += [f"{col}_filled", f"has_{col}"]
        else:
            numeric_feats.append(col)

    freshness_dummies = pd.get_dummies(work["freshness_tier"], prefix="freshness")
    X_all = pd.concat([work[numeric_feats], freshness_dummies], axis=1)
    y_all = (work["trend_direction"] == "down").astype(int)
    return X_all, y_all

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

def run_split(grouped, n_repeats=8):
    scores = []
    for split_i in range(n_repeats):
        if grouped:
            gss = GroupShuffleSplit(n_splits=1, test_size=TEST_SIZE, random_state=RANDOM_SEED + split_i)
            train_idx, test_idx = next(gss.split(df, groups=df["client_id"]))
        else:
            ss = ShuffleSplit(n_splits=1, test_size=TEST_SIZE, random_state=RANDOM_SEED + split_i)
            train_idx, test_idx = next(ss.split(df))

        X_all, y_all = build_features(df, train_idx)
        X_train, X_test = X_all.iloc[train_idx], X_all.iloc[test_idx]
        y_train, y_test = y_all.iloc[train_idx], y_all.iloc[test_idx]

        scaler = StandardScaler().fit(X_train)
        X_train_scaled = scaler.transform(X_train)
        X_test_scaled = scaler.transform(X_test)

        clf = LogisticRegression(max_iter=2000, random_state=RANDOM_SEED)
        clf.fit(X_train_scaled, y_train)
        proba = clf.predict_proba(X_test_scaled)[:, 1]
        p50 = precision_at_k(proba, y_test.values, 50)
        scores.append(p50)
    return np.array(scores)

print("Running BEFORE: naive random row split, Logistic Regression, 8 repeats...")
before_scores = run_split(grouped=False, n_repeats=8)

print("Running AFTER: client-grouped split (matches w05's design), Logistic Regression, 8 repeats...")
after_scores = run_split(grouped=True, n_repeats=8)

print("\n=== BEFORE vs AFTER: precision@50, Logistic Regression, 8 splits each, test_size=0.2 ===")
print(f"BEFORE (random row split):    mean={before_scores.mean():.3f}  sd={before_scores.std():.3f}")
print(f"AFTER  (client-grouped split): mean={after_scores.mean():.3f}  sd={after_scores.std():.3f}")
gap = before_scores.mean() - after_scores.mean()
print(f"\nGap (memorization signal): {gap:+.3f}")

os.makedirs(DRIVE_OUTPUTS, exist_ok=True)
pd.DataFrame({
    "split": range(8),
    "before_random_split": before_scores,
    "after_grouped_split": after_scores,
}).to_csv(f"{DRIVE_OUTPUTS}/split_before_after.csv", index=False)
print(f"\nSaved to {DRIVE_OUTPUTS}/split_before_after.csv")


Running BEFORE: naive random row split, Logistic Regression, 8 repeats...
Running AFTER: client-grouped split (matches w05's design), Logistic Regression, 8 repeats...

=== BEFORE vs AFTER: precision@50, Logistic Regression, 8 splits each, test_size=0.2 ===
BEFORE (random row split):    mean=0.828  sd=0.024
AFTER  (client-grouped split): mean=0.735  sd=0.060

Gap (memorization signal): +0.093

Saved to /content/drive/MyDrive/work/outputs/split_before_after.csv


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

Running the full attack checklist against **w05's actual Logistic Regression feature set** (17 columns per w05's own printed `X_train.shape`), including a positive control that deliberately reintroduces `trend_pct` to confirm the harness actually detects a leak when one exists.

In [25]:
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import GroupShuffleSplit
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score

if not os.path.exists("data/raw/content_refresh_anonymized.csv"):
    os.chdir("/content")
    if not os.path.isdir("FlyRankAI-ML-Internship"):
        import subprocess
        subprocess.run([
            "git", "clone", "--depth", "1",
            "https://github.com/haida-ishtiaq/FlyRankAI-ML-Internship"
        ], check=True)
    os.chdir("FlyRankAI-ML-Internship")

RANDOM_SEED = 42
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# --- Checklist: no label-derived or sibling columns in the w05 feature list ---
w05_feature_cols = {"impressions_90d", "ctr", "avg_position", "engagement_rate", "scroll_rate",
                     "days_since_last_update", "content_age_days", "word_count",
                     "freshness_tier", "days_with_impressions"}
label_derived_cols = {"trend_direction", "trend_pct"}  # Correct: only columns that exist
overlap = w05_feature_cols & label_derived_cols
print("=== Checklist: label-derived / sibling columns ===")
print("Overlap (must be empty):", overlap)
assert len(overlap) == 0, "LEAKAGE: a label-derived column is in the feature set!"
print("PASSED\n")

# --- Checklist: no product/decision-flag columns as features ---
possible_product_flags = {"age_tier", "position_tier", "impression_tier", "competition_level",
                           "word_count_tier", "char_count_tier"}
flags_in_features = w05_feature_cols & possible_product_flags
print("=== Checklist: product/decision flags as features ===")
print("Overlap (must be empty):", flags_in_features)
assert len(flags_in_features) == 0, "A tier/decision column is being used as a feature!"
print("PASSED -- freshness_tier IS used, but it is a bucketed raw feature, not an external decision flag.\n")

# --- Checklist: avg_position = 0 means "no data" ---
print("=== Checklist: avg_position handling ===")
avg_pos_zero_count = (df["avg_position"] == 0).sum()
print(f"Rows with avg_position=0 (no data): {avg_pos_zero_count}")
print("These are handled via has_position_data flag and median imputation")
print("PASSED\n")

# --- Checklist: scroll_rate can exceed 100 ---
print("=== Checklist: scroll_rate can exceed 100 (expected per data dictionary) ===")
scroll_rate_max = df["scroll_rate"].max()
print(f"scroll_rate max: {scroll_rate_max:.2f}")
if scroll_rate_max > 100:
    print("scroll_rate exceeds 100 - this is expected (different measurement systems)")
else:
    print("scroll_rate within expected range")
print("PASSED\n")

# --- Positive control: deliberately add trend_pct, confirm the harness detects it ---
print("=== Positive control ===")
work = df.copy()
work["avg_position"] = work["avg_position"].replace(0, np.nan)
work["avg_position"] = work["avg_position"].fillna(work["avg_position"].median())
y = (work["trend_direction"] == "down").astype(int)

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=RANDOM_SEED)
train_idx, test_idx = next(gss.split(work, groups=work["client_id"]))

honest_cols = ["impressions_90d", "ctr", "avg_position", "engagement_rate",
               "scroll_rate", "days_since_last_update", "content_age_days", "days_with_impressions"]
X_honest = work[honest_cols].fillna(0)
Xs_honest = StandardScaler().fit(X_honest.iloc[train_idx]).transform(X_honest)
clf_honest = LogisticRegression(max_iter=2000, random_state=RANDOM_SEED)
clf_honest.fit(Xs_honest[train_idx], y.iloc[train_idx])
auc_honest = roc_auc_score(y.iloc[test_idx], clf_honest.predict_proba(Xs_honest[test_idx])[:, 1])

# FIX: Fill NaN values in trend_pct before adding it
X_leaky = X_honest.copy()
X_leaky["trend_pct"] = work["trend_pct"].fillna(0)
Xs_leaky = StandardScaler().fit(X_leaky.iloc[train_idx]).transform(X_leaky)
clf_leaky = LogisticRegression(max_iter=2000, random_state=RANDOM_SEED)
clf_leaky.fit(Xs_leaky[train_idx], y.iloc[train_idx])
auc_leaky = roc_auc_score(y.iloc[test_idx], clf_leaky.predict_proba(Xs_leaky[test_idx])[:, 1])

print(f"Honest ROC-AUC: {auc_honest:.4f}")
print(f"Leaky ROC-AUC (trend_pct reintroduced): {auc_leaky:.4f}")
print(f"Jump: {auc_leaky - auc_honest:+.4f}")
assert auc_leaky > auc_honest + 0.05, "Positive control FAILED -- harness did not detect a deliberate leak!"
print("PASSED\n")

print("=== Base rate ===")
print(f"Base rate: {y.mean():.3f} -- every metric elsewhere in this repo should be read next to this.")
print("\n=== Note on data interpretation ===")
print("Rate columns (ctr, engagement_rate, scroll_rate) are ×100 percentages per flyrank-data skill.")
print("ctr=0.76 means 0.76%, not 76% - this should be considered when interpreting feature importance.")

=== Checklist: label-derived / sibling columns ===
Overlap (must be empty): set()
PASSED

=== Checklist: product/decision flags as features ===
Overlap (must be empty): set()
PASSED -- freshness_tier IS used, but it is a bucketed raw feature, not an external decision flag.

=== Checklist: avg_position handling ===
Rows with avg_position=0 (no data): 1205
These are handled via has_position_data flag and median imputation
PASSED

=== Checklist: scroll_rate can exceed 100 (expected per data dictionary) ===
scroll_rate max: 300.00
scroll_rate exceeds 100 - this is expected (different measurement systems)
PASSED

=== Positive control ===
Honest ROC-AUC: 0.5783
Leaky ROC-AUC (trend_pct reintroduced): 0.9991
Jump: +0.4208
PASSED

=== Base rate ===
Base rate: 0.542 -- every metric elsewhere in this repo should be read next to this.

=== Note on data interpretation ===
Rate columns (ctr, engagement_rate, scroll_rate) are ×100 percentages per flyrank-data skill.
ctr=0.76 means 0.76%, not 76% - t

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

| Notebook | Original claim | Audited rewrite | Why |
|---|---|---|---|
| w02 (Section 2) | "The label...is an observed outcome...not a hand-written rule." | `is_refresh_opportunity` is a three-condition threshold rule (`impressions_90d>=500 AND content_age_days>=180 AND impressions_drop>0`), which the `framing-ml-problems` skill explicitly defines as the thing to avoid: "a label that comes from someone's rule means your model learns the rule, not the world." This target was never carried into w04/w05, which used `trend_direction` instead -- worth documenting as a deliberate pivot, not silently dropped. | The skill's own core distinction (observed vs. defined) was stated correctly in prose but violated in the actual code for this label. |
| w05 (Section 4) | "The problem is hard -- even Random Forest can't beat random guessing beyond K=20." | Random Forest scores below the base rate (0.511) at **both** K=20 (0.40) and K=50 (0.48), and loses to the hand-written rule baseline at K=20 (0.40 vs 0.45). Logistic Regression, evaluated on the identical split and metric, beats the base rate and the rule at every K (0.75/0.74/0.73). The honest statement is: Random Forest underperforms on this dataset; Logistic Regression is the validated winner and is the model this repo's downstream work (w06, w07) is built on. | The original phrasing implies the *task* is hard for all methods, when the comparison table shows one method working clearly and the other not -- exactly the case the skill says to name explicitly ("report both; that IS the finding"), not soften into a general difficulty statement. |
| w03 (missing section) | N/A -- omission, not overclaim | Add a one-sentence "Output" statement per `writing-data-contracts/SKILL.md` Section 5: e.g. "This analysis hands the editorial team a page-level feature table, ready for ranking, with `trend_direction`/`trend_pct` withheld as the label to predict." | The skill requires this as its own numbered section; w03 currently omits it entirely rather than stating it briefly. |

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

In [31]:
import os
import pandas as pd

assert os.path.exists(f"{DRIVE_OUTPUTS}/split_before_after.csv"), "Before/after split comparison missing!"
comp = pd.read_csv(f"{DRIVE_OUTPUTS}/split_before_after.csv")
assert {"before_random_split", "after_grouped_split"}.issubset(comp.columns)
assert len(comp) == 8
print("Self-check PASSED.")

Self-check PASSED.
